# Real Qwen decode projection check
The setup cell adds pinned Transformers dependencies to the isolated Python 3.12 environment. First use downloads several GB of model files.

The script loads Qwen2.5-1.5B in FP16 (checkpoint config is BF16). For each of three fixed prompts, it runs prefill, chooses the highest-scoring next token, and feeds that token through a cached decode step. A temporary hook captures the input and output of layer 0 mlp.up_proj. The hook is removed immediately.

It checks captured output, PyTorch linear replay, and frozen rows1 GEMV against an FP32 reference computed from the loaded FP16 tensors. Only passing cases are timed. The benchmark replays the isolated projection; loading, full-model execution and capture are outside timing. This is not a whole-model speedup measurement. Raw samples and model revision are saved.


In [ ]:
import sys
import subprocess
from pathlib import Path

repo = Path('/kaggle/working/fastinfer')
python = '/kaggle/working/fastinfer-t4-py312/bin/python'
assert Path(python).exists(), 'Use the existing Python 3.12 GPU environment.'
assert (repo / 'kernels/triton_gemv.py').exists(), 'Existing GEMV checkout required.'
subprocess.run([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', python,
    'torch==2.5.1', 'triton==3.1.0', 'numpy==1.26.4',
    'transformers==4.46.3', 'accelerate==1.1.1', 'safetensors==0.4.5',
], check=True)

# Register this checkout in the isolated interpreter used by the benchmark.
subprocess.run([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', python,
    '--no-deps', '--editable', str(repo),
], check=True)
subprocess.run([
    python, '-c',
    'import fastinfer; from fastinfer.kernels.triton_gemv import gemv; '
    'print("fastinfer import OK:", fastinfer.__file__)',
], cwd='/kaggle/working', check=True)


In [ ]:
files = {'benchmarks/qwen_linear_check.py': '"""Check a frozen GEMV on real Qwen weights and captured decode activations."""\n\nimport argparse\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nimport runpy\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n\nROOT = Path(__file__).resolve().parents[1]\nMODEL = "Qwen/Qwen2.5-1.5B"\nPROMPTS = (\n    "The capital of France is",\n    "Explain why the sky appears blue in one sentence.",\n    "def add(a, b):\\n    return",\n)\n\n\nclass NonfiniteTensorError(RuntimeError):\n    """A numerical failure with JSON-safe diagnostic details."""\n\n    def __init__(self, details):\n        self.details = details\n        super().__init__(f"Nonfinite tensor at {details[\'stage\']}: {details}")\n\n\ndef require_finite(value, stage):\n    import torch\n\n    if not torch.isfinite(value).all().item():\n        raise NonfiniteTensorError({\n            "stage": stage,\n            "shape": list(value.shape),\n            "dtype": str(value.dtype),\n            "nan_count": torch.isnan(value).sum().item(),\n            "inf_count": torch.isinf(value).sum().item(),\n            "elements": value.numel(),\n        })\n\n\n@contextmanager\ndef trace_finite_modules(model, phase):\n    """Check leaf-module boundaries; hooks are removed before any timing.\n\n    Inspect the first tensor only: attention masks can legitimately contain -inf.\n    This locates the first observed failure, not necessarily its internal operation.\n    """\n    import torch\n\n    handles = []\n\n    def check(value, stage):\n        if isinstance(value, (tuple, list)) and value:\n            value = value[0]\n        if isinstance(value, torch.Tensor) and value.is_floating_point():\n            require_finite(value, stage)\n\n    try:\n        for name, child in model.named_modules():\n            if not name or any(child.children()):\n                continue\n            stage = f"{phase}:{name}"\n            handles.append(child.register_forward_pre_hook(\n                lambda module, inputs, stage=stage: check(inputs, stage + ".input")\n            ))\n            handles.append(child.register_forward_hook(\n                lambda module, inputs, output, stage=stage: check(output, stage + ".output")\n            ))\n        yield\n    finally:\n        for handle in handles:\n            handle.remove()\n\n\ndef capture_decode_projection(model, projection, encoded):\n    """Run prefill, then capture this layer\'s input/output for one decoded token."""\n    import torch\n\n    captured = []\n\n    def capture(module, inputs, output):\n        captured.append((inputs[0].detach().clone(), output.detach().clone()))\n\n    with torch.inference_mode():\n        with trace_finite_modules(model, "prefill"):\n            prefill = model(**encoded, use_cache=True, return_dict=True)\n        require_finite(prefill.logits, "prefill.logits")\n        token = prefill.logits[:, -1, :].argmax(dim=-1, keepdim=True)\n        mask = torch.cat((encoded["attention_mask"], torch.ones_like(token)), dim=1)\n        # Register only after prefill: the captured input must be a single token.\n        handle = projection.register_forward_hook(capture)\n        try:\n            with trace_finite_modules(model, "decode"):\n                decode = model(\n                    input_ids=token,\n                    attention_mask=mask,\n                    past_key_values=prefill.past_key_values,\n                    use_cache=True,\n                    return_dict=True,\n                )\n            require_finite(decode.logits, "decode.logits")\n        finally:\n            handle.remove()\n\n    if len(captured) != 1 or captured[0][0].shape[:2] != (1, 1):\n        raise RuntimeError("expected exactly one [batch=1, token=1, hidden] capture")\n    return captured[0][0], captured[0][1], token.item()\n\n\ndef time_pair(a, weight, helpers, gemv, round_index):\n    import torch\n    import torch.nn.functional as F\n\n    operations = {\n        "pytorch_linear": lambda x, w: F.linear(x, w),\n        "gemv_rows1": lambda x, w: gemv(x, w, block_rows=1, num_warps=4),\n    }\n    stream = torch.cuda.Stream()\n    stream.wait_stream(torch.cuda.current_stream())\n    with torch.cuda.stream(stream):\n        for _ in range(10):\n            for operation in operations.values():\n                operation(a, weight)\n    torch.cuda.current_stream().wait_stream(stream)\n    torch.cuda.synchronize()\n    graphs, outputs = {}, {}\n    names = list(operations)\n    for name in (names if round_index % 2 == 0 else names[::-1]):\n        graphs[name], outputs[name] = helpers["capture_graph"](operations[name], a, weight, 100)\n    for _ in range(10):\n        for graph in graphs.values():\n            graph.replay()\n    torch.cuda.synchronize()\n    samples = {name: [] for name in names}\n    for repetition in range(30):\n        order = names if (repetition + round_index) % 2 == 0 else names[::-1]\n        for name in order:\n            start = torch.cuda.Event(enable_timing=True)\n            end = torch.cuda.Event(enable_timing=True)\n            start.record()\n            graphs[name].replay()\n            end.record()\n            end.synchronize()\n            samples[name].append(start.elapsed_time(end) * 1000 / 100)\n    return {name: helpers["summarize"](values) for name, values in samples.items()}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--revision", default="main")\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n\n    import torch\n    import torch.nn.functional as F\n    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n    from fastinfer.kernels.triton_gemv import gemv\n\n    if not torch.cuda.is_available():\n        parser.error("a CUDA GPU is required")\n    torch.set_float32_matmul_precision("highest")\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True\n    helpers = runpy.run_path(str(ROOT / "benchmarks/matmul_compare.py"))\n\n    config = AutoConfig.from_pretrained(MODEL, revision=args.revision, trust_remote_code=False)\n    revision = config._commit_hash\n    if not revision:\n        raise RuntimeError("could not resolve model revision for reproducibility")\n    print(f"Loading {MODEL} revision {revision} in FP16...", flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=revision, trust_remote_code=False)\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL,\n        revision=revision,\n        torch_dtype=torch.float16,\n        low_cpu_mem_usage=True,\n        use_safetensors=True,\n        trust_remote_code=False,\n        attn_implementation="eager",\n    ).to("cuda").eval()\n    projection = model.model.layers[0].mlp.up_proj\n    weight = projection.weight.detach()\n    if weight.shape != (8960, 1536) or weight.dtype != torch.float16:\n        raise RuntimeError(f"unexpected projection weights: {weight.shape}, {weight.dtype}")\n    if projection.bias is not None or not weight.is_contiguous():\n        raise RuntimeError("expected bias-free projection with contiguous stored weights")\n\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n    destination = args.output or ROOT / "results/raw" / f"qwen_real_linear_{stamp}.json"\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    report = {\n        "benchmark": "qwen_captured_decode_linear",\n        "model": MODEL,\n        "resolved_revision": revision,\n        "projection": "model.layers.0.mlp.up_proj",\n        "model_dtype": "float16",\n        "checkpoint_config_dtype": str(config.torch_dtype),\n        "attention_implementation": "eager",\n        "shape": {"m": 1, "k": 1536, "n": 8960},\n        "weight_stride": list(weight.stride()),\n        "tile": {"block_rows": 1, "num_warps": 4},\n        "allow_fp16_reduced_precision_reduction": True,\n        "timing_mode": "cuda_graph_replay_amortized",\n        "graph_ops": 100, "warmup": 10, "graph_warmup": 10, "repetitions": 30,\n        "measurement_order": "alternating_pytorch_gemv_pairs",\n        "accuracy_reference": "float32_matmul_of_loaded_fp16_weights_and_activation",\n        "rtol": .01, "atol": .01,\n        "scope": "Isolated projection replay on real weights and captured decode inputs. "\n                 "Model stays resident. Model loading, prefill, decode and capture excluded "\n                 "from timing. No model replacement or end-to-end speedup measured.",\n        "kernel_sha256": hashlib.sha256((ROOT / "kernels/triton_gemv.py").read_bytes()).hexdigest(),\n        "script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        "environment": json.loads(subprocess.check_output(\n            [sys.executable, str(ROOT / "scripts/environment_report.py")], cwd=ROOT, text=True)),\n        "cases": [],\n    }\n    with torch.inference_mode():\n        for index, prompt in enumerate(PROMPTS):\n            encoded = tokenizer(prompt, return_tensors="pt").to("cuda")\n            try:\n                require_finite(weight, "projection.weight")\n                activation, model_output, token = capture_decode_projection(model, projection, encoded)\n                require_finite(activation, "captured_activation")\n                require_finite(model_output, "captured_model_output")\n                a = activation.reshape(1, -1).contiguous()\n                captured_output = model_output.reshape(1, -1)\n                reference = F.linear(a.float(), weight.float())\n                require_finite(reference, "float32_reference")\n            except NonfiniteTensorError as error:\n                report["cases"].append({\n                    "prompt": prompt,\n                    "status": "nonfinite_model_or_reference",\n                    "diagnostics": error.details,\n                    "timing_skipped": True,\n                })\n                destination.write_text(json.dumps(report, indent=2) + "\\n")\n                print(f"Diagnosis saved: {destination}", flush=True)\n                raise\n            actual = gemv(a, weight, block_rows=1, num_warps=4)\n            pytorch_output = F.linear(a, weight)\n            accuracy = {\n                name: helpers["accuracy_against_fp32"](value, reference)\n                for name, value in (\n                    ("captured_model_output", captured_output),\n                    ("pytorch_linear", pytorch_output),\n                    ("gemv_rows1", actual),\n                )\n            }\n            capture_match = torch.allclose(pytorch_output, captured_output, rtol=.01, atol=.01)\n            passed = capture_match and all(v["mismatched_elements"] == 0 for v in accuracy.values())\n            case = {\n                "prompt": prompt, "prompt_tokens": encoded.input_ids.shape[1],\n                "decoded_token_id": token, "captured_input_shape": list(activation.shape),\n                "input_stride": list(a.stride()), "accuracy": accuracy,\n                "replay_matches_captured_output": capture_match,\n                "all_accuracy_checks_passed": passed,\n            }\n            report["cases"].append(case)\n            print(f"\\nPrompt {index + 1}: {prompt!r}", flush=True)\n            for name, values in accuracy.items():\n                print(f"  {name}: {values[\'mismatched_elements\']} mismatches, "\n                      f"max error {values[\'max_abs_error\']:.6f}", flush=True)\n            if passed:\n                del reference, actual, pytorch_output\n                torch.cuda.synchronize()\n                case["implementations"] = time_pair(a, weight, helpers, gemv, index)\n                pt = case["implementations"]["pytorch_linear"]["latency_us"]["median"]\n                tt = case["implementations"]["gemv_rows1"]["latency_us"]["median"]\n                print(f"  PyTorch {pt:.2f} us | GEMV {tt:.2f} us | PT/GEMV {pt / tt:.3f}x", flush=True)\n            else:\n                print("  Timing skipped: accuracy check failed; see saved diagnostics.", flush=True)\n            destination.write_text(json.dumps(report, indent=2) + "\\n")\n    print(f"Saved: {destination}")\n\n\nif __name__ == "__main__":\n    main()\n', 'tests/test_qwen_linear_capture.py': '"""Check decode-only capture and hook cleanup without downloading a model."""\n\nimport importlib.util\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport unittest\n\nimport torch\n\n\npath = Path(__file__).resolve().parents[1] / "benchmarks/qwen_linear_check.py"\nspec = importlib.util.spec_from_file_location("qwen_linear_check", path)\nmodule = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(module)\n\n\nclass TinyModel(torch.nn.Module):\n    def __init__(self, fail_decode=False):\n        super().__init__()\n        self.projection = torch.nn.Linear(3, 2, bias=False)\n        self.cache = object()\n        self.calls = []\n        self.fail_decode = fail_decode\n\n    def forward(self, input_ids, attention_mask, past_key_values=None, **kwargs):\n        self.calls.append((input_ids.clone(), attention_mask.clone(), past_key_values))\n        x = input_ids.float().unsqueeze(-1).expand(-1, -1, 3)\n        self.projection(x)\n        if past_key_values is not None and self.fail_decode:\n            raise RuntimeError("decode failure")\n        logits = torch.zeros((*input_ids.shape, 5))\n        logits[..., 3] = 1\n        return SimpleNamespace(logits=logits, past_key_values=self.cache)\n\n\nclass DecodeCaptureTest(unittest.TestCase):\n    def setUp(self):\n        self.encoded = {\n            "input_ids": torch.tensor([[1, 2]]),\n            "attention_mask": torch.tensor([[1, 1]]),\n        }\n\n    def test_captures_only_decode_and_passes_cache(self):\n        model = TinyModel()\n        activation, output, token = module.capture_decode_projection(\n            model, model.projection, self.encoded\n        )\n        self.assertEqual(token, 3)\n        self.assertEqual(tuple(activation.shape), (1, 1, 3))\n        torch.testing.assert_close(activation, torch.full((1, 1, 3), 3.0))\n        with torch.inference_mode():\n            torch.testing.assert_close(output, model.projection(activation))\n        self.assertEqual(len(model.calls), 2)\n        self.assertIs(model.calls[1][2], model.cache)\n        self.assertEqual(model.calls[1][1].tolist(), [[1, 1, 1]])\n        self.assertEqual(len(model.projection._forward_hooks), 0)\n\n    def test_nonfinite_projection_stops_prefill_and_removes_hooks(self):\n        model = TinyModel()\n        with torch.no_grad():\n            model.projection.weight.fill_(float("nan"))\n        with self.assertRaises(module.NonfiniteTensorError) as caught:\n            module.capture_decode_projection(model, model.projection, self.encoded)\n        self.assertEqual(caught.exception.details["stage"], "prefill:projection.output")\n        self.assertEqual(len(model.calls), 1)\n        self.assertEqual(len(model.projection._forward_hooks), 0)\n        self.assertEqual(len(model.projection._forward_pre_hooks), 0)\n\n    def test_nonfinite_logits_rejected_before_token_selection(self):\n        model = TinyModel()\n        def corrupt_logits(module_, inputs, output):\n            output.logits.fill_(float("inf"))\n        handle = model.register_forward_hook(corrupt_logits)\n        try:\n            with self.assertRaises(module.NonfiniteTensorError) as caught:\n                module.capture_decode_projection(model, model.projection, self.encoded)\n            self.assertEqual(caught.exception.details["stage"], "prefill.logits")\n            self.assertEqual(len(model.calls), 1)\n        finally:\n            handle.remove()\n\n    def test_removes_hook_when_decode_fails(self):\n        model = TinyModel(fail_decode=True)\n        with self.assertRaisesRegex(RuntimeError, "decode failure"):\n            module.capture_decode_projection(model, model.projection, self.encoded)\n        self.assertEqual(len(model.projection._forward_hooks), 0)\n\n\nif __name__ == "__main__":\n    unittest.main()\n'}
for name, source in files.items():
    (repo / name).write_text(source)
subprocess.run([python, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_qwen_linear_capture.py', '-v'], cwd=repo, check=True)
subprocess.run([python, 'benchmarks/qwen_linear_check.py'], cwd=repo, check=True)
